# Events - Silver Transformation

## Parameters

In [0]:
dbutils.widgets.text(
    name="environment",
    defaultValue="dev",
    label="Environment"
)

environment = dbutils.widgets.get("environment").strip().lower()

if environment not in ("dev", "test", "prod"):
    raise ValueError(
        f"Unsupported environment: {environment}. Expected dev, test, or prod."
    )

## Setup

In [0]:
from pyspark.sql.functions import timestamp_millis

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.retailrocket_events"
target_table = f"{catalog}.silver.retailrocket_events"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Transform to Silver

In [0]:
silver_df = (
    bronze_df
    .withColumnRenamed("visitorid", "visitor_id")
    .withColumnRenamed("event", "event_type")
    .withColumnRenamed("itemid", "item_id")
    .withColumnRenamed("transactionid", "transaction_id")
)

In [0]:
silver_df = silver_df.withColumn(
    "event_timestamp",
    timestamp_millis("timestamp")
)

## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)